# Repository Diff and Sync

This notebook builds simplified repository diff and sync, mirroring how
ATProto MST (Merkle Search Tree) generates diffs and how the firehose
communicates changes between repositories.

**What you'll learn:**
- How MST snapshots represent repository state
- How diff operations (add, update, delete) are generated
- How diffs are applied to synchronize repositories

**Estimated Time:** 15-20 minutes

## Step 1: MST Snapshot

A real MST is a tree structure with depth determined by key hashing.
Here we simplify to a flat dictionary of key→CID pairs. The key insight
is that comparing two snapshots produces a list of diff operations.

In [ ]:
@interface MSTSnapshot : NSObject
@property (nonatomic, strong) NSMutableDictionary *entries;
- (void)put:(NSString *)key cid:(NSString *)cid;
- (NSString *)get:(NSString *)key;
- (void)remove:(NSString *)key;
- (NSMutableArray *)allEntries;
- (NSString *)description;
@end

@implementation MSTSnapshot
- (instancetype)init {
    self = [super init];
    if (self) { _entries = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)put:(NSString *)key cid:(NSString *)cid {
    [_entries setObject:cid forKey:key];
}
- (NSString *)get:(NSString *)key {
    return [_entries objectForKey:key];
}
- (void)remove:(NSString *)key {
    [_entries removeObjectForKey:key];
}
- (NSMutableArray *)allEntries {
    NSMutableArray *result = [NSMutableArray array];
    for (NSString *key in _entries) {
        [result addObject:@{@"key": key, @"cid": [_entries objectForKey:key]}];
    }
    return result;
}
- (NSString *)description {
    return [NSString stringWithFormat:@"MSTSnapshot(%d entries)", [_entries count]];
}
@end

MSTSnapshot *snap = [[MSTSnapshot alloc] init];
[snap put:@"app.bsky.feed.post/abc" cid:@"bafyrei1111"];
[snap put:@"app.bsky.feed.post/def" cid:@"bafyrei2222"];
[snap put:@"app.bsky.actor.profile/self" cid:@"bafyrei3333"];
NSLog(@"%@", [snap description]);
NSLog(@"Post abc: %@", [snap get:@"app.bsky.feed.post/abc"]);

## Step 2: Repo Diff

`MST.diffFrom:` compares two MST versions and returns diff operations.
Each operation is: add (new key), update (changed CID), or delete (removed key).
This is what the firehose broadcasts as commit events.

In [ ]:
@interface RepoDiff : NSObject
- (NSMutableArray *)diffFrom:(MSTSnapshot *)oldSnap to:(MSTSnapshot *)newSnap;
@end

@implementation RepoDiff
- (NSMutableArray *)diffFrom:(MSTSnapshot *)oldSnap to:(MSTSnapshot *)newSnap {
    NSMutableArray *ops = [NSMutableArray array];
    
    // Find adds and updates: keys in newSnap
    for (NSString *key in newSnap.entries) {
        NSString *newCid = [newSnap get:key];
        NSString *oldCid = [oldSnap get:key];
        if (oldCid == nil) {
            // New key → add
            [ops addObject:@{
                @"action": @"add",
                @"key": key,
                @"cid": newCid
            }];
        } else if (![newCid isEqualToString:oldCid]) {
            // Changed CID → update
            [ops addObject:@{
                @"action": @"update",
                @"key": key,
                @"prevCid": oldCid,
                @"cid": newCid
            }];
        }
    }
    
    // Find deletes: keys in oldSnap but not in newSnap
    for (NSString *key in oldSnap.entries) {
        if ([newSnap get:key] == nil) {
            [ops addObject:@{
                @"action": @"delete",
                @"key": key,
                @"prevCid": [oldSnap get:key]
            }];
        }
    }
    
    return ops;
}
@end

## Step 3: Repo Sync

Applying a diff to a snapshot produces a new snapshot. This mirrors
how repository sync works — a consumer receives diff operations and
applies them to bring their local state up to date.

In [ ]:
@interface RepoSync : NSObject
- (MSTSnapshot *)applyDiff:(NSMutableArray *)diffOps toSnapshot:(MSTSnapshot *)snapshot;
@end

@implementation RepoSync
- (MSTSnapshot *)applyDiff:(NSMutableArray *)diffOps toSnapshot:(MSTSnapshot *)snapshot {
    MSTSnapshot *newSnap = [[MSTSnapshot alloc] init];
    // Copy existing entries
    for (NSString *key in snapshot.entries) {
        [newSnap put:key cid:[snapshot get:key]];
    }
    // Apply diff operations
    for (int i = 0; i < [diffOps count]; i++) {
        NSDictionary *op = [diffOps objectAtIndex:i];
        NSString *action = [op objectForKey:@"action"];
        NSString *key = [op objectForKey:@"key"];
        if ([action isEqualToString:@"add"] || [action isEqualToString:@"update"]) {
            [newSnap put:key cid:[op objectForKey:@"cid"]];
        } else if ([action isEqualToString:@"delete"]) {
            [newSnap remove:key];
        }
    }
    return newSnap;
}
@end

## Step 4: Full Diff and Sync Demo

Create two snapshots, diff them, verify the operations, apply the diff
to the old snapshot, and verify it matches the new one.

In [ ]:
// Old snapshot
MSTSnapshot *old = [[MSTSnapshot alloc] init];
[old put:@"app.bsky.feed.post/abc" cid:@"bafyrei1111"];
[old put:@"app.bsky.feed.post/def" cid:@"bafyrei2222"];
[old put:@"app.bsky.actor.profile/self" cid:@"bafyrei3333"];

// New snapshot (1 add, 1 update, 1 delete)
MSTSnapshot *new_ = [[MSTSnapshot alloc] init];
[new_ put:@"app.bsky.feed.post/abc" cid:@"bafyrei1111"];  // unchanged
[new_ put:@"app.bsky.feed.post/def" cid:@"bafyrei9999"];  // updated CID
[new_ put:@"app.bsky.feed.post/xyz" cid:@"bafyrei4444"];  // new key
// profile/self deleted

// Compute diff
RepoDiff *differ = [[RepoDiff alloc] init];
NSMutableArray *ops = [differ diffFrom:old to:new_];
NSLog(@"Diff operations: %d", [ops count]);
for (int i = 0; i < [ops count]; i++) {
    NSDictionary *op = [ops objectAtIndex:i];
    NSLog(@"  %@ %@", [op objectForKey:@"action"], [op objectForKey:@"key"]);
}

// Apply diff to old snapshot
RepoSync *syncer = [[RepoSync alloc] init];
MSTSnapshot *synced = [syncer applyDiff:ops toSnapshot:old];
NSLog(@"Synced: %@", [synced description]);

// Verify synced matches new
NSLog(@"Post abc: %@ (expect bafyrei1111)", [synced get:@"app.bsky.feed.post/abc"]);
NSLog(@"Post def: %@ (expect bafyrei9999)", [synced get:@"app.bsky.feed.post/def"]);
NSLog(@"Post xyz: %@ (expect bafyrei4444)", [synced get:@"app.bsky.feed.post/xyz"]);
NSLog(@"Profile: %@ (expect nil)", [synced get:@"app.bsky.actor.profile/self"]);

## Exercise

Add `reverseDiff:` to `RepoDiff` that inverts a diff — adds become deletes,
deletes become adds, and updates swap `prevCid` and `cid`. This is useful for
rolling back changes.

Hint: loop over the operations and create inverse operations for each.

In [ ]:
// Exercise: implement reverseDiff:
// Your code here...

## Solution

In [ ]:
// Solution: reverseDiff inverts each operation
@interface RepoDiff2 : NSObject
- (NSMutableArray *)reverseDiff:(NSMutableArray *)diffOps;
@end

@implementation RepoDiff2
- (NSMutableArray *)reverseDiff:(NSMutableArray *)diffOps {
    NSMutableArray *reversed = [NSMutableArray array];
    for (int i = 0; i < [diffOps count]; i++) {
        NSDictionary *op = [diffOps objectAtIndex:i];
        NSString *action = [op objectForKey:@"action"];
        NSString *key = [op objectForKey:@"key"];
        if ([action isEqualToString:@"add"]) {
            // Add becomes delete
            [reversed addObject:@{
                @"action": @"delete",
                @"key": key,
                @"prevCid": [op objectForKey:@"cid"]
            }];
        } else if ([action isEqualToString:@"delete"]) {
            // Delete becomes add
            [reversed addObject:@{
                @"action": @"add",
                @"key": key,
                @"cid": [op objectForKey:@"prevCid"]
            }];
        } else if ([action isEqualToString:@"update"]) {
            // Update swaps CIDs
            [reversed addObject:@{
                @"action": @"update",
                @"key": key,
                @"prevCid": [op objectForKey:@"cid"],
                @"cid": [op objectForKey:@"prevCid"]
            }];
        }
    }
    return reversed;
}
@end

// Test it: reverse the diff and apply it to new_ to get back to old
RepoDiff2 *rd2 = [[RepoDiff2 alloc] init];
NSMutableArray *revOps = [rd2 reverseDiff:ops];
NSLog(@"Reversed operations: %d", [revOps count]);
for (int i = 0; i < [revOps count]; i++) {
    NSDictionary *op = [revOps objectAtIndex:i];
    NSLog(@"  %@ %@", [op objectForKey:@"action"], [op objectForKey:@"key"]);
}

// Apply reverse diff to new_ should produce old
MSTSnapshot *rolledBack = [syncer applyDiff:revOps toSnapshot:new_];
NSLog(@"Post def after rollback: %@ (expect bafyrei2222)", [rolledBack get:@"app.bsky.feed.post/def"]);
NSLog(@"Profile after rollback: %@ (expect bafyrei3333)", [rolledBack get:@"app.bsky.actor.profile/self"]);

## Next Steps

You have learned about ATProto's core data structures and protocols. Continue to explore how these concepts apply when building a PDS.